In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import re




from astroquery.gaia import Gaia
from astroquery.vizier import Vizier
import requests
from bs4 import BeautifulSoup
import json

from datetime import datetime

import os

import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
#defining global variable path to files and data, it is cvalled thropughout modify this path to match your local path
path = './'

## Project Objective

Project Question = *Does the Sun have a twin or a closely related cousin?*

Yhe goal of this project is to see can we determine if there another star in the universe that is identical or similar to our Sun, that we know of and have measurements, metrics for.

We will tak astronmical data from avariety of diffeent public sources. 

#### 1) Dataset - NASA >  Stellar Hosts

Data set taken from https://exoplanetarchive.ipac.caltech.edu/ 

We will down lai the 'Stellar Hosts' dataset. This dataset is 1 row per star. It is determined by the search for exoplanets, sok each star will have 0 to > 0 number of planets orbitting it.

In [11]:
# after a bit of playing around tihe the url format we get the url for csv format for the entire stellar hosts dataset
# we download in csv format
# This uses their preferred TAP API, like a sql command to directly query the database, and we get it in a clean csv.
url = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?query=select+*+from+stellarhosts&format=csv"
stellar_host_df = pd.read_csv(url)

stellar_host_df.shape

# create a lcoal copy in csv as had issue with server going offline and access data. so save lcoal copy in case needed
# wil put a time stampe on the filename for uniqueness
datetime_stamp = datetime.now().strftime("%d%m%y")
sh_csv_file_name = f"stellarhosts_{datetime_stamp}.csv"
stellar_host_df.to_csv(sh_csv_file_name, index=False)
print(f"saved {sh_csv_file_name}")

# NOTE th is nto one row per star >> all sceintific measurements are kept and sources are different, so needs to be filered....
# may add update a paramater, i.e. update mass of a star , added in a new row, old row kept...

saved stellarhosts_091225.csv


In [12]:
stellar_host_df.head()

stellar_host_df.shape

(46870, 136)

Need to get the uniqwue values for GAIA_IDs
we will sue those to download other datasets. 
Other datasets are to big, in TBs in siuze, and this will allow us filter just what we need, reduce downlaod time and make ti more manageable

In [13]:
# get gaia columns from the datafram
gaia_cols = [col for col in stellar_host_df.columns if'gaia' in col.lower()]

print(f"gaia columns  : {gaia_cols}")

gaia columns  : ['sy_gaiamag', 'sy_gaiamagerr1', 'sy_gaiamagerr2', 'gaia_dr2_id', 'gaia_dr3_id']


view first 10 valeus for "gaia_dr2_id", "gaia_dr3_id" just oc heck the types of values

In [15]:
stellar_host_df[["gaia_dr2_id", "gaia_dr3_id"]].head(10)

,gaia_dr2_id,gaia_dr3_id
0,Gaia DR2 2128190453050802048,Gaia DR3 2128190453050802048
1,Gaia DR2 2105930840143687680,Gaia DR3 2105930840143687680
2,Gaia DR2 2077595394707557120,Gaia DR3 2077595394707557120
3,Gaia DR2 2085724496490595584,Gaia DR3 2085724496490595584
4,Gaia DR2 2129158435598210816,Gaia DR3 2129158435598210816
5,Gaia DR2 2086414268238649984,Gaia DR3 2086414268238649984
6,Gaia DR2 2052582432887909376,Gaia DR3 2052582432887909376
7,Gaia DR2 2052942900903300352,Gaia DR3 2052942900903300352
8,Gaia DR2 2130211080541825024,Gaia DR3 2130211080541825024
9,Gaia DR2 2128079230577866880,Gaia DR3 2128079230577866880


see how many of the numebrs match across both "gaia_dr2_id", "gaia_dr3_id" 

In [18]:
compare = stellar_host_df[["gaia_dr2_id", "gaia_dr3_id"]].head(10)
compare["match"] = compare["gaia_dr2_id"] == compare["gaia_dr3_id"]
print(compare)

                    gaia_dr2_id                   gaia_dr3_id  match
0  Gaia DR2 2128190453050802048  Gaia DR3 2128190453050802048  False
1  Gaia DR2 2105930840143687680  Gaia DR3 2105930840143687680  False
2  Gaia DR2 2077595394707557120  Gaia DR3 2077595394707557120  False
3  Gaia DR2 2085724496490595584  Gaia DR3 2085724496490595584  False
4  Gaia DR2 2129158435598210816  Gaia DR3 2129158435598210816  False
5  Gaia DR2 2086414268238649984  Gaia DR3 2086414268238649984  False
6  Gaia DR2 2052582432887909376  Gaia DR3 2052582432887909376  False
7  Gaia DR2 2052942900903300352  Gaia DR3 2052942900903300352  False
8  Gaia DR2 2130211080541825024  Gaia DR3 2130211080541825024  False
9  Gaia DR2 2128079230577866880  Gaia DR3 2128079230577866880  False


want to get the GAIA ID numebrs, minus the strings
Problem
need to match onyl yrh gaia IDs 8numbers) so can downlaod onyl the data we wnat from gaia downlaod
DR3 and DR4 are nto 100% identical m and this is as expected as ID get updated as data is updated sources are split, uydpated data changes thigns etc so it loks like our data is intact and we have ths striped gaiaa ID

In [106]:
stellar_host_df_2 = stellar_host_df.copy()

#problemnumbertyep19digitnumbercompareasastring?
# need to use numpyarraytosuenjkmber
# pandas os displying 19 digit numbers a
print(stellar_host_df_2['gaia_dr2_id'].head(10).tolist())
print(stellar_host_df_2['gaia_dr3_id'].head(10).tolist())

# http://regex101.com
# $ start at the end of thestring, only match the last part,
# smatch 10 to 20 digits in a row, ignore the space, so we don't get the 2 from DR2, so contiguous, between 10 and 20 characters and at the end of the string 
# stop at space
stellar_host_df_2['dr2_num'] = stellar_host_df_2['gaia_dr2_id'].astype(str).str.extract(r'(\d{10,20})', expand=False)
stellar_host_df_2['dr3_num'] = stellar_host_df_2['gaia_dr3_id'].astype(str).str.extract(r'(\d{10,20})', expand=False)

print(stellar_host_df_2['dr2_num'].head())
print(stellar_host_df_2['dr3_num'].head())

stellar_host_df_2.shape

# stellar_host_df_2['dr2_equals_dr3'] = stellar_host_df_2['dr2_num'] == stellar_host_df_2['dr3_num']

# stellar_host_df_2 = stellar_host_df_2.dropna(subset=['dr3_num'])stellar_host_df_2['dr2_equals_dr3'].value_counts()

['Gaia DR2 2128190453050802048', 'Gaia DR2 2105930840143687680', 'Gaia DR2 2077595394707557120', 'Gaia DR2 2085724496490595584', 'Gaia DR2 2129158435598210816', 'Gaia DR2 2086414268238649984', 'Gaia DR2 2052582432887909376', 'Gaia DR2 2052942900903300352', 'Gaia DR2 2130211080541825024', 'Gaia DR2 2128079230577866880']
['Gaia DR3 2128190453050802048', 'Gaia DR3 2105930840143687680', 'Gaia DR3 2077595394707557120', 'Gaia DR3 2085724496490595584', 'Gaia DR3 2129158435598210816', 'Gaia DR3 2086414268238649984', 'Gaia DR3 2052582432887909376', 'Gaia DR3 2052942900903300352', 'Gaia DR3 2130211080541825024', 'Gaia DR3 2128079230577866880']
0    2128190453050802048
1    2105930840143687680
2    2077595394707557120
3    2085724496490595584
4    2129158435598210816
Name: dr2_num, dtype: object
0    2128190453050802048
1    2105930840143687680
2    2077595394707557120
3    2085724496490595584
4    2129158435598210816
Name: dr3_num, dtype: object


(46870, 138)

In [ ]:
#  drop any rows where the dr3_num column contains Na values, missing data
stellar_host_df_2 = stellar_host_df_2.dropna(subset=['dr3_num'])

# keep onyl rows where have a full digit match, so no spaces, dots, commas erc 
stellar_host_df_2 = stellar_host_df_2[stellar_host_df_2['dr3_num'].str.fullmatch(r'\d{10,20}')]

In [107]:
# get shape again to see if any difference from earlier shape
#no change 
stellar_host_df_2.shape



(46870, 138)

craete alist just containign the gaia IDs

In [33]:
gaia_dr3_ids = stellar_host_df_2['dr3_num'].astype(str).tolist()
print(len(gaia_dr3_ids))

46870


In [47]:
# find all DR3 catalogs in vizier

cats = Vizier.find_catalogs('Gaia DR3')

cats

OrderedDict([('I/324', </>),
             ('I/337', </>),
             ('I/345', </>),
             ('I/347', </>),
             ('I/350', </>),
             ('I/352', </>),
             ('I/355', </>),
             ('I/356', </>),
             ('I/357', </>),
             ('I/358', </>),
             ('I/359', </>),
             ('I/360', </>),
             ('I/361', </>),
             ('II/350', </>),
             ('IV/36', </>),
             ('VI/137', </>),
             ('VI/145', </>),
             ('J/A+A/523/A48', </>),
             ('J/A+A/674/A25', </>)])

get a lost of theavailable tables in the 1/355 DR3 catalog

In [48]:
Vizier.ROW_LIMIT = 0

# we choose the I/355 catalog as per theri website >> https://cdsarc.cds.unistra.fr/viz-bin/cat/I/355 this contains allw e need
tables = Vizier.get_catalogs("I/355")
print(len(tables))

# get a list tables within the 
for i, t in enumerate(tables):
    print(f"\nTable index {i}:")
    print("  Columns:", t.columns[:10])


15

Table index 0:
  Columns: <TableColumns names=('RA_ICRS','DE_ICRS','Source','e_RA_ICRS','e_DE_ICRS','Plx','e_Plx','PM','pmRA','e_pmRA')>

Table index 1:
  Columns: <TableColumns names=('Source','RA_ICRS','DE_ICRS','PQSO','PGal','Pstar','PWD','Pbin','Teff','logg')>

Table index 2:
  Columns: <TableColumns names=('Source','RA_ICRS','DE_ICRS','SolID','Teff','logg','[M/H]','Dist','A0','AG')>

Table index 3:
  Columns: <TableColumns names=('SolID','HPId','HPlevel','A0','e_A0','b_A0','B_A0','Ntracers','foptHP','Status')>

Table index 4:
  Columns: <TableColumns names=('SolID','HPId','A0','e_A0','Ntracers','Status','HPlevel')>

Table index 5:
  Columns: <TableColumns names=('SolID','SOMID','NeuId','NeuRowIdx','NeuColIdx','Class','Gmag','BPmag','RPmag','pmRA')>

Table index 6:
  Columns: <TableColumns names=('SolID','NeuId','NeuRowIdx','NeuColIdx','XPFlux','XPlambda','XPFluxTemp')>

Table index 7:
  Columns: <TableColumns names=('Source','RA_ICRS','DE_ICRS','TransitID','TimeG','FG','e_FG',

In [57]:
# check the data in the source columns i the tables to see fi they are 19 digit numbers


# print(tables[0]['Source'][:10])

# print(tables[2]['Source'][:20])
# print(type(tables[0]['Source'][0]))
# print(type(tables[2]['Source'][0]))

# Convert to string for filtering
main_df['Source'] = main_df['Source'].astype(str)
ap_df['Source']   = ap_df['Source'].astype(str)

# Keep only 19-digit Gaia IDs
main_df = main_df[main_df['Source'].str.fullmatch(r'\d{19}')]
ap_df   = ap_df[ap_df['Source'].str.fullmatch(r'\d{19}')]



NameError: name 'main_df' is not defined

In [ ]:
Vizier.ROW_LIMIT = -1
main_table = 1
ap_table = 2

# 1. Main source table
v_main = Vizier(columns=["source_id", "ra", "dec", "parallax", "phot_g_mean_mag",
                         "phot_bp_mean_mag", "phot_rp_mean_mag", "bp_rp", "ruwe"])
main = v_main.get_catalogs("I/355/gaiadr3")[0].to_pandas()

# 2. GSP-Phot AP table
v_ap = Vizier(columns=["source_id", "teff_gspphot", "logg_gspphot", "mh_gspphot",
                       "radius_gspphot", "lum_gspphot", "mass_gspphot"])
ap = v_ap.get_catalogs("I/355/gaiadr3/ap")[0].to_pandas()

# # 3. GSP-Spec table (if needed)
# v_spec = Vizier(columns=["source_id", "teff_gspspec", "logg_gspspec", "feh_gspspec"])
# spec = v_spec.get_catalogs("I/355/gaiadr3/spectra")[0].to_pandas()

get gaia main table data

In [45]:
from astroquery.vizier import Vizier

Vizier.ROW_LIMIT = 0

print("MAIN TABLE:")
main = Vizier.get_catalogs("I/355/gaiadr3")[0]
print(main.columns[:10])

print("\nAP TABLE (GSP-Phot):")
ap = Vizier.get_catalogs("I/355/gaiadr3ap")[0]
print(ap.columns[:10])

print("\nGSP-SPEC TABLE:")
spec = Vizier.get_catalogs("I/355/gaiadr3spec")[0]
print(spec.columns[:10])





MAIN TABLE:
<TableColumns names=('RA_ICRS','DE_ICRS','Source','e_RA_ICRS','e_DE_ICRS','Plx','e_Plx','PM','pmRA','e_pmRA')>

AP TABLE (GSP-Phot):


IndexError: list index out of range

downlaod ap table

In [102]:
from astroquery.vizier import Vizier
import pandas as pd

Vizier.ROW_LIMIT = -1

catalog_id = "I/355"
ap_table_idx = 2

# take one test ID from your list
test_id = stellar_host_df_2['dr3_num'].astype(str).iloc[0]

v = Vizier()

result = v.query_constraints(
    catalog=catalog_id,
    Source=str(test_id),
    table=ap_table_idx
)

if len(result) == 0:
    print("No match for this ID in AP table.")
else:
    df = result[0].to_pandas()
    df['Source'] = df['Source'].astype(str)
    print(df.head())




      RA_ICRS    DE_ICRS               Source  e_RA_ICRS  e_DE_ICRS    Plx  \
0  293.577796  46.736295  2128190453050802048     0.1847     0.2072  0.088   

    e_Plx     PM   pmRA  e_pmRA  ...  TYC2            URAT1  \
0  0.2083  2.595 -0.224   0.256  ...        URAT1-684292625   

               AllWISE  APASS9       GSC23  RAVE5             2MASS  RAVE6  \
0  J193418.66+464410.9    <NA>  N2JE060725         19341866+4644108          

      RAJ2000    DEJ2000  
0  293.577798  46.736307  

[1 rows x 57 columns]


download AP data table

In [67]:
target_ids = stellar_host_df_2['dr3_num'].astype(str).tolist()
print("IDs to query:", len(target_ids))

Vizier.ROW_LIMIT = -1        # allow all matched rows
catalog_id = "I/355"         # Gaia DR3
ap_table_idx = 2             # Table 2 = Astrophysical Parameters

v = Vizier()

print("Querying Gaia DR3 AP table...")

result = v.query_constraints(
    catalog=catalog_id,
    Source=target_ids,        # your Gaia DR3 ID list
    table=ap_table_idx
)

if len(result) == 0:
    raise ValueError("No AP rows returned. Check ID list formatting.")

ap_df = result[0].to_pandas()

IDs to query: 45132
Querying Gaia DR3 AP table...


KeyboardInterrupt: 

In [ ]:
ap = Vizier.get_catalogs("I/355/gaiadr3/ap")[0].to_pandas()

In [70]:
# see if tap service is available

from astroquery.utils.tap.core import TapPlus

tap = TapPlus(url="http://tapvizier.cds.unistra.fr/TAPVizieR/tap")

# simple test query:
job = tap.launch_job("SELECT TOP 5 source_id FROM gaiadr3.gaia_source")
print(job.get_results())


ConnectionResetError: [WinError 10054] An existing connection was forcibly closed by the remote host

In [108]:
from astroquery.utils.tap.core import TapPlus
import pandas as pd

# ---------------------------------------------------------
# 1. Connect to ARI Gaia TAP mirror
# ---------------------------------------------------------
tap = TapPlus(url="https://gaia.ari.uni-heidelberg.de/tap")

# ---------------------------------------------------------
# 2. Prepare your Gaia DR3 source_ids
# ---------------------------------------------------------
# MUST be strings; convert from your dataframe
id_list = stellar_host_df_2["dr3_num"].astype(str).tolist()

print("Number of Gaia IDs:", len(id_list))

# Build ID list as a comma-separated string
id_str = ",".join(id_list)

# ---------------------------------------------------------
# 3. Build ADQL query for the AP table
# ---------------------------------------------------------
query = f"""
SELECT source_id, teff_gspphot, logg_gspphot, mh_gspphot,
       distance_gspphot, a0_gspphot, ag_gspphot
FROM gaiadr3.astrophysical_parameters
WHERE source_id IN ({id_str})
"""

print("Submitting query...")

# ---------------------------------------------------------
# 4. Run query ASYNC (fast)
# ---------------------------------------------------------
job = tap.launch_job_async(query)
ap_result = job.get_results()
ap_df = ap_result.to_pandas()

print("Rows returned:", len(ap_df))

# ---------------------------------------------------------
# 5. Save the data
# ---------------------------------------------------------
ap_df.to_csv("gaia_dr3_ap_ari.csv", index=False)
print("Saved AP data to gaia_dr3_ap_ari.csv")


Number of Gaia IDs: 46870
Submitting query...


TimeoutError: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond

In [109]:
# import pandas as pd

# url1 = "https://dr8.lamost.org/catalogue/params/lrs_stellar_params.csv.gz"
# lamost_df = pd.read_csv(url1)
# print(lamost_df.head())


# import socket
# socket.gethostbyname("dr8.lamost.org")

# import socket
# socket.gethostbyname("dr8.lamost.org")
# socket.gethostbyname("gaia.ari.uni-heidelberg.de")
# socket.gethostbyname("cds.unistra.fr")

import socket

hosts = [
    "dr8.lamost.org",
    "www.lamost.org",
    "gaia.ari.uni-heidelberg.de",
    "cds.unistra.fr",

    # TAP servers
    "tap.lamost.org",
    "gaia.ari.uni-heidelberg.de",  # Gaia TAP is same domain
    "gea.esac.esa.int",            # ESA Gaia Archive
    "simbad.u-strasbg.fr",         # CDS TAP
    "vizier.u-strasbg.fr",         # VizieR TAP
]

for host in hosts:
    try:
        print(f"{host:30s} → {socket.gethostbyname(host)}")
    except Exception as e:
        print(f"{host:30s} → ERROR: {e}")



dr8.lamost.org                 → ERROR: [Errno 11001] getaddrinfo failed
www.lamost.org                 → 159.226.170.43
gaia.ari.uni-heidelberg.de     → 129.206.112.47
cds.unistra.fr                 → 130.79.128.30
tap.lamost.org                 → ERROR: [Errno 11001] getaddrinfo failed
gaia.ari.uni-heidelberg.de     → 129.206.112.47
gea.esac.esa.int               → 193.147.152.106
simbad.u-strasbg.fr            → 130.79.128.4
vizier.u-strasbg.fr            → 130.79.128.13


In [110]:
import requests

url = "http://www.lamost.org/dr8/static/data/stars/lrs_stellar_params.csv.gz"
r = requests.get(url)
open("lrs_stellar_params.csv.gz", "wb").write(r.content)


757

In [111]:
import requests
from requests.exceptions import SSLError, ConnectionError

def download_lamost(url_https, output_filename):
    url_http = url_https.replace("https://", "http://")
    
    print(f"\nAttempting download (HTTPS): {url_https}")
    try:
        r = requests.get(url_https, timeout=10)
        r.raise_for_status()
        print("✓ HTTPS download succeeded")
        with open(output_filename, "wb") as f:
            f.write(r.content)
        return
    
    except SSLError:
        print("✗ SSL certificate problem — switching to HTTP")
    except ConnectionError:
        print("✗ HTTPS connection failed — switching to HTTP")
    except Exception as e:
        print(f"✗ HTTPS failed: {e} — switching to HTTP")

    print(f"Attempting download (HTTP): {url_http}")
    r = requests.get(url_http, timeout=10)
    r.raise_for_status()
    print("✓ HTTP download succeeded")
    with open(output_filename, "wb") as f:
        f.write(r.content)

# EXAMPLE: LRS Stellar Parameters
download_lamost(
    "https://dr.lamost.org/catalogue/params/lrs_stellar_params.csv.gz",
    "lrs_stellar_params.csv.gz"
)



Attempting download (HTTPS): https://dr.lamost.org/catalogue/params/lrs_stellar_params.csv.gz
✗ SSL certificate problem — switching to HTTP
Attempting download (HTTP): http://dr.lamost.org/catalogue/params/lrs_stellar_params.csv.gz
✓ HTTP download succeeded


In [89]:
import pandas as pd

url = "https://vizier.cds.unistra.fr/viz-bin/asu-tsv?-source=V/166A"
df = pd.read_csv(url, sep='\t', comment='#')
df.head(), df.shape




ParserError: Error tokenizing data. C error: Expected 10 fields in line 4148, saw 11


In [ ]:
df = pd.read_csv("lrs_stellar_params.csv.gz", compression="gzip")


In [112]:
# test gaia tap server


url = "https://gea.esac.esa.int/tap-server/tap/capabilities"

try:
    r = requests.get(url, timeout=5)
    print("Status code:", r.status_code)

    if r.status_code == 200:
        print("ESA Gaia TAP: ✅ ONLINE")
    else:
        print("ESA Gaia TAP: ⚠️ Responded but not healthy")

except requests.exceptions.RequestException as e:
    print("ESA Gaia TAP: ❌ OFFLINE or UNREACHABLE")
    print("Error:", e)


Status code: 503
ESA Gaia TAP: ⚠️ Responded but not healthy


In [113]:
# test gaoa tap server
import requests
url = "https://gea.esac.esa.int/tap-server/tap/capabilities"
print(requests.get(url).status_code)



503


In [114]:
#connect to lamost server

import pandas as pd
df = pd.read_csv("https://dr8.lamost.org/catalogue/params/lrs_stellar_params.csv.gz")
print(df.head())


URLError: <urlopen error [Errno 11001] getaddrinfo failed>

In [115]:
#connect to heidelberg missor tap server

from astroquery.utils.tap.core import TapPlus

tap = TapPlus(url="https://gaia.ari.uni-heidelberg.de/tap")

job = tap.launch_job("SELECT TOP 5 source_id FROM gaiadr3.gaia_source")
print(job.get_results())


TimeoutError: [WinError 10060] A connection attempt failed because the connected party did not properly respond after a period of time, or established connection failed because connected host has failed to respond

In [117]:
# test tap serer connections
import socket
print(socket.gethostbyname("dr8.lamost.org"))
print(socket.gethostbyname("gaia.ari.uni-heidelberg.de"))
print(socket.gethostbyname("cds.unistra.fr"))


gaierror: [Errno 11001] getaddrinfo failed

In [118]:
# connect and download from FTP

#ftp://cdsarc.cds.unistra.fr/pub/cats/I/355/gaiadr3/
#ftp://cdsarc.cds.unistra.fr/pub/cats/I/355/gaiadr3_ap/

import pandas as pd

ap_url = "https://cdsarc.cds.unistra.fr/ftp/cats/I/355/gaiadr3_ap/ap.csv.gz"
ap = pd.read_csv(ap_url)


HTTPError: HTTP Error 404: Not Found

In [119]:
# test https socket to ARI

import socket
import ssl

hostname = "gaia.ari.uni-heidelberg.de"
port = 443

context = ssl.create_default_context()
sock = socket.create_connection((hostname, port), timeout=5)
ssock = context.wrap_socket(sock, server_hostname=hostname)

print("CONNECTED OK:", ssock.version())

ssock.close()


TimeoutError: timed out

In [120]:
import pandas as pd

url_ap = "https://cdsarc.cds.unistra.fr/ftp/cats/I/355/gaiadr3_ap/ap.csv.gz"
ap = pd.read_csv(url_ap)

print(ap.head())
print(len(ap))



HTTPError: HTTP Error 404: Not Found

In [ ]:
from astroquery.utils.tap import TapPlus
import pandas as pd
from io import StringIO
import math

# Connect to VizieR TAP service
vizier_tap = TapPlus(url="https://tapvizier.cds.unistra.fr/TAP")


def fetch_gaia_ap_vizier(id_list, chunk_size=1500):
    dfs = []
    n = len(id_list)
    chunks = [id_list[i:i+chunk_size] for i in range(0, n, chunk_size)]

    for idx, chunk in enumerate(chunks, 1):
        print(f"Running chunk {idx}/{len(chunks)} ...")

        id_str = ",".join(str(int(x)) for x in chunk)

        query = f"""
        SELECT *
        FROM "I/355/gaiadr3ap"
        WHERE source_id IN ({id_str})
        """

        job = vizier_tap.launch_job(query=query, format="csv")
        df = job.get_results().to_pandas()
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

# Run query using your Gaia DR3 ID list
gaia_ap_vizier_df = fetch_gaia_ap_vizier(dr3_list)
print("Downloaded rows:", gaia_ap_vizier_df.shape)


In [ ]:
from astroquery.gaia import Gaia
import pandas as pd

def fetch_gaia_ap_chunked(dr3_ids, chunk_size=1500):
    dfs = []
    for i in range(0, len(dr3_ids), chunk_size):
        chunk = dr3_ids[i:i+chunk_size]
        id_str = ",".join(str(x) for x in chunk)

        query = f"""
        SELECT source_id, 
               teff_gspphot, logg_gspphot, mh_gspphot,
               radius_gspphot, lum_gspphot,
               ag_gspphot, ebpminrp_gspphot
        FROM gaiadr3.astrophysical_parameters
        WHERE source_id IN ({id_str})
        """

        print(f"Chunk {i//chunk_size+1}...")
        job = Gaia.launch_job_async(query)
        df = job.get_results().to_pandas()
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)

gaia_ap_df = fetch_gaia_ap_chunked(gaia_id_dr3_list)
print(gaia_ap_df.shape)


In [ ]:
# columsn to download
f
from this table >> I/355/paramp ( 1590932717 rows ) (positions) 1D astrophysical parameters produced by the Apsis >> on this page https://tapvizier.u-strasbg.fr/adql/?I/355 >> what colmns ared best to get ??

Source,
Teff,
logg,
[Fe/H],
Rad,
Lum-Flame,
Mass-Flame,
Age-Flame,
A0,
AG,
E(BP-RP),
GMAG,
Pstar